# CardioIA Fase 4 — Notebook 1: Pré-processamento e Organização (Parte 1)

**Aluno:** Diogo Zequini · RM 565535 · Turma 2TIAOA
**Dataset:** NIH Chest X-ray14 — recorte **Cardiomegaly vs. No Finding**

> ⚠️ **Protótipo acadêmico (FIAP).** Não é dispositivo médico, não foi validado clinicamente e não deve ser usado para diagnóstico real.

Este notebook seleciona o subconjunto, aplica o **split por `Patient ID`** (anti-vazamento)
e gera o manifesto + EDA. Rode no **Kaggle** com o dataset `nih-chest-xrays/data` anexado.


In [1]:
# ===== CardioIA Fase 4 - Etapa 1: subset + split por paciente (anti-vazamento) =====
# Roda no Kaggle (dataset nih-chest-xrays/data anexado). Saidas em /kaggle/working.
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEED = 565535
OUT = Path("/kaggle/working")

INPUT = Path("/kaggle/input")
csv = next(INPUT.rglob("Data_Entry_2017.csv"), None)
assert csv, "Data_Entry_2017.csv nao encontrado."
print("CSV:", csv)

df = pd.read_csv(csv)
print("Linhas no CSV:", len(df), "| Pacientes unicos:", df["Patient ID"].nunique())

df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
df = df[df["Patient Age"].between(0, 120)].copy()

tem_cardio = df["Finding Labels"].str.split("|").map(lambda x: "Cardiomegaly" in x)
pos = df[tem_cardio].copy(); pos["target"], pos["target_label"] = 1, "Cardiomegaly"
neg = df[df["Finding Labels"].eq("No Finding")].copy(); neg["target"], neg["target_label"] = 0, "No Finding"
print("Disponivel -> Cardiomegaly:", len(pos), "| No Finding:", len(neg))

N = min(1200, len(pos), len(neg))
subset = pd.concat([pos.sample(N, random_state=SEED), neg.sample(N, random_state=SEED)], ignore_index=True)
subset = subset.sample(frac=1, random_state=SEED).reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
tr_idx, tmp_idx = next(gss.split(subset, groups=subset["Patient ID"]))
train, tmp = subset.iloc[tr_idx].copy(), subset.iloc[tmp_idx].copy()
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=SEED + 1)
v_idx, t_idx = next(gss2.split(tmp, groups=tmp["Patient ID"]))
val, test = tmp.iloc[v_idx].copy(), tmp.iloc[t_idx].copy()
train["split"], val["split"], test["split"] = "train", "val", "test"
manifest = pd.concat([train, val, test], ignore_index=True)

S = {s: set(g["Patient ID"]) for s, g in manifest.groupby("split")}
assert not (S["train"] & S["val"]) and not (S["train"] & S["test"]) and not (S["val"] & S["test"])
print("\nOK: sem vazamento de paciente entre os splits.")

todas = {p.name: str(p) for p in INPUT.rglob("*.png")}
manifest["filepath"] = manifest["Image Index"].map(todas)
print("Imagens sem caminho resolvido:", int(manifest["filepath"].isna().sum()))
manifest = manifest.dropna(subset=["filepath"]).reset_index(drop=True)

manifest.to_csv(OUT / "data_manifest.csv", index=False)
print("\nManifesto salvo:", OUT / "data_manifest.csv", "| Total imagens:", len(manifest))

ct_cls = pd.crosstab(manifest["split"], manifest["target_label"])
ct_pac = manifest.groupby("split")["Patient ID"].nunique()
ct_view = pd.crosstab(manifest["split"], manifest["View Position"])
ct_gen = pd.crosstab(manifest["Patient Gender"], manifest["target_label"])

print("\nImagens por split x classe:\n", ct_cls)
print("\nPacientes unicos por split:\n", ct_pac)
print("\nView Position por split:\n", ct_view)
print("\nGenero x classe:\n", ct_gen)

resumo = {
    "seed": SEED,
    "n_por_classe": int(N),
    "total_imagens": int(len(manifest)),
    "pacientes_por_split": ct_pac.to_dict(),
    "split_x_classe": ct_cls.to_dict(),
    "vazamento": "0 (interseccao vazia de Patient ID)",
}
(OUT / "eda_resumo.json").write_text(json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nResumo salvo:", OUT / "eda_resumo.json")

CSV: /kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv
Linhas no CSV: 112120 | Pacientes unicos: 30805
Disponivel -> Cardiomegaly: 2776 | No Finding: 60353

OK: sem vazamento de paciente entre os splits.
Imagens sem caminho resolvido: 0

Manifesto salvo: /kaggle/working/data_manifest.csv | Total imagens: 2400

Imagens por split x classe:
 target_label  Cardiomegaly  No Finding
split                                 
test                   176         169
train                  857         850
val                    167         181

Pacientes unicos por split:
 split
test      281
train    1306
val       280
Name: Patient ID, dtype: int64

View Position por split:
 View Position   AP    PA
split                   
test           139   206
train          705  1002
val            127   221

Genero x classe:
 target_label    Cardiomegaly  No Finding
Patient Gender                          
F                        641         499
M                        559     

## Resultados (execução real no Kaggle)

- CSV: **112.120 imagens / 30.805 pacientes**. Disponível: Cardiomegaly **2.776**, No Finding **60.353**.
- Subset balanceado: **2.400 imagens (1.200/classe)**, todos os caminhos resolvidos.
- **Split por paciente, sem vazamento** (interseção vazia provada):
  - treino 1.707 imgs / 1.306 pacientes · val 348 / 280 · test 345 / 281
- Artefato: `data_manifest.csv` (proveniência completa).
